# Focused Direct and Total Site

**Learning outcome:** Apply focused direct and total site through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Core  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** plot

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** How do local process targets and their equivalent global HRAT differ from indirect and Total Site opportunities?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Establish the site-wide reference

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("pulp_mill.json", project_name="Site")
complete = problem.target.all_heat_integration()
summary = problem.summary_frame()
summary

## Step 2: Run explicit integration scopes and inspect the equivalent HRAT

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
direct = problem.target.direct_heat_integration(
    zone="Bleaching", period_id="0"
)
requested_recovery_kw = float(direct.heat_recovery_target)
cached_results_before_inverse = problem.results
recovery_approach = problem.target.heat_recovery_approach_temperature(
    heat_recovery={"value": requested_recovery_kw, "unit": "kW"},
    zone="Bleaching",
    period_id="0",
)
ordinary_target_preserved = (
    problem.results is cached_results_before_inverse
)
recovery_approach_summary = recovery_approach.model_dump(mode="json")
recovery_approach_summary["ordinary_target_preserved"] = (
    ordinary_target_preserved
)
indirect = problem.target.indirect_heat_integration()
total_site = problem.target.total_site_heat_integration()
total_site_profiles = problem.plot.total_site_profiles()
site_utility_curve = problem.plot.site_utility_grand_composite_curve()

## Review the result

Inspect the requested and achieved recovery, approach temperature, thermodynamic limit, residual, status, and iteration count together. The ordinary target identity demonstrates that the inverse call is non-mutating. Then compare the process summary with the Total Site profiles and utility grand composite curve; the site views reveal opportunities that are not visible within one process zone.

In [ ]:
from IPython.display import display

display(summary)
display(recovery_approach_summary)
display(total_site_profiles)
display(site_utility_curve)

## Interpret the result

Compare like-for-like duties and retain the zone and period with every focused result. For an interior request, the inverse service returns the greatest feasible global HRAT whose calculated recovery still meets the request. It is a process-level composite-curve spacing, not an exchanger EMAT; Total Site curves describe utility-system opportunities, not individual exchanger matches.

## Adapt this template

Change `zone="Bleaching"` to a path from your zone tree and compare the same scope across scenarios.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.